# DFT XC skeleton 二阶导数分解 (B3LYP, GGA)

对标 `06-2-decomp_de_xc_tpss0.ipynb`，但 B3LYP 是 hybrid GGA，无 tau 贡献。我们仍采用与 TPSS0 一致的策略：

- **fxc 部分**：直接使用 rho 导数（每原子方向给出 `[4, ngrids]` 的一阶密度导数），与 fxc 核 `[4, 4, ngrids]` 直接缩并。
- **vxc 部分**（non-diagonal）：先给出原子非依赖的 `[3, 3, nao, nao]` 二阶分量矩阵，再依双原子加权求和。
- **vxc 部分**（diagonal）：通过 ao 三阶导数与 ao_dm0 缩并。

相比 TPSS0，这里不再有 tau (deriv=4) 相关项。


In [1]:
from pyscf import gto, dft, lib
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
import sys
sys.path.append("..")

from pyhessref.nimatmul.becke_partition import becke_partition
from pyhessref.nimatmul import rks as rks_nimatmul

In [3]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [4]:
mf = dft.RKS(mol, xc="B3LYP").density_fit()
dat0 = np.load("nh3_r_b3lyp.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [5]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mocc = mo_coeff[:, mo_occ > 0]
dm0 = mocc @ mocc.T * 2
natm = mol.natm
nao = mol.nao
aoslices = mol.aoslice_by_atom()
ni = dft.numint.NumInt()

In [6]:
# grids = dft.grid.Grids(mol)
# grids.coords = coords = dat0["grid_coords"]
# grids.weights = weights = dat0["grid_weights"]
# ngrids = len(weights)

In [7]:
grids = dft.gen_grid.Grids(mol)
grids.build(sort_grids=False)
coords = grids.coords
weights = grids.weights
ngrids = len(weights)

In [8]:
# Reference de_vxc from 06-4: this is what we want to reproduce.
de_ks_ref = np.load("nh3_r_b3lyp_decomp.npz")["de_vxc"]
print("de_vxc_ref shape:", de_ks_ref.shape)
print("de_vxc_ref fp:   ", lib.fp(de_ks_ref))

de_vxc_ref shape: (4, 4, 3, 3)
de_vxc_ref fp:    -0.8985828139605962


In [9]:
ao = ni.eval_ao(mol, grids.coords, deriv=3)
rho = ni.eval_rho2(mol, ao[:4], mo_coeff, mo_occ, xctype="GGA")
print("rho shape:", rho.shape)

rho shape: (4, 43328)


In [10]:
xc_eff = ni.eval_xc_eff(mf.xc, rho, deriv=2, xctype="GGA")
vxc = xc_eff[1]  # shape [4, ngrid]
fxc = xc_eff[2]  # shape [4, 4, ngrid]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)

vxc shape: (4, 43328) fxc shape: (4, 4, 43328)


In [11]:
TX, TY, TZ = 0, 1, 2
O = 0
X, Y, Z = 1, 2, 3
XX, XY, XZ = 4, 5, 6
YX, YY, YZ = 5, 7, 8
ZX, ZY, ZZ = 6, 8, 9
XXX, XXY, XXZ, XYY, XYZ, XZZ = 10, 11, 12, 13, 14, 15
YYY, YYZ, YZZ, ZZZ = 16, 17, 18, 19

In [12]:
ao_dm0 = ao @ dm0
ao_dm0.shape

(20, 43328, 49)

### fxc contribution

In [13]:
# B3LYP (GGA): no tau, drho has 4 components (RHO, GRAD_X, GRAD_Y, GRAD_Z)
drho = np.zeros((natm, 3, 4, ngrids))
for A in range(natm):
    _, _, p0, p1 = aoslices[A]
    slc = slice(p0, p1)
    ao_slc = ao[:, :, slc]
    ao_dm0_slc = ao_dm0[:, :, slc]
    DERIV_COMPONENTS = [
        # RHO part
        [(TX, 0), (X, O)],
        [(TY, 0), (Y, O)],
        [(TZ, 0), (Z, O)],
        # SIGMA part (bra deriv 2)
        [(TX, X), (XX, O)],
        [(TX, Y), (XY, O)],
        [(TX, Z), (XZ, O)],
        [(TY, X), (YX, O)],
        [(TY, Y), (YY, O)],
        [(TY, Z), (YZ, O)],
        [(TZ, X), (ZX, O)],
        [(TZ, Y), (ZY, O)],
        [(TZ, Z), (ZZ, O)],
        # SIGMA part (bra deriv 1, ket deriv 1)
        [(TX, X), (X, X)],
        [(TX, Y), (X, Y)],
        [(TX, Z), (X, Z)],
        [(TY, X), (Y, X)],
        [(TY, Y), (Y, Y)],
        [(TY, Z), (Y, Z)],
        [(TZ, X), (Z, X)],
        [(TZ, Y), (Z, Y)],
        [(TZ, Z), (Z, Z)],
    ]
    for ((t, v), (cbra, cket)) in DERIV_COMPONENTS:
        drho[A, t, v] -= np.einsum("gu, gu -> g", ao_slc[cbra], ao_dm0_slc[cket])
# scale symmetric coeff: all 4 components (RHO + SIGMA) get *2
drho *= 2

In [14]:
lib.fp(drho)

np.float64(-3951374.647722333)

In [15]:
de_fxc = np.einsum("g, Atxg, xyg, Bsyg -> ABts", weights, drho, fxc, drho)
print(lib.fp(de_fxc))

-21.24987446516278


### dao_vxc_diag contribution

In [16]:
# --- dao_vxc_diag (GGA: no tau) --- #
dao_vxc_diag = np.zeros((6, nao))  # 6 denotes xx, xy, xz, yy, yz, zz
wv = weights * vxc  # [4, ngrids]

# Contribution 1: ao[i+4]^T @ (wv[0]*ao[0] + wv[1]*ao[1] + wv[2]*ao[2] + wv[3]*ao[3])
aow_diag = (np.einsum("gu, g -> gu", ao_dm0[0], wv[0])
          + np.einsum("gu, g -> gu", ao_dm0[1], wv[1])
          + np.einsum("gu, g -> gu", ao_dm0[2], wv[2])
          + np.einsum("gu, g -> gu", ao_dm0[3], wv[3]))
for idx, its in enumerate([XX, XY, XZ, YY, YZ, ZZ]):
    dao_vxc_diag[idx] += 2 * np.einsum("gu, gu -> u", ao[its], aow_diag)

# Contribution 2 (GGA triple-derivative part)
TRIPLE_DERIV_DIAG = [
    [XXX, XXY, XXZ],  # xx
    [XXY, XYY, XYZ],  # xy
    [XXZ, XYZ, XZZ],  # xz
    [XYY, YYY, YYZ],  # yy
    [XYZ, YYZ, YZZ],  # yz
    [XZZ, YZZ, ZZZ],  # zz
]
for idx, (i3x, i3y, i3z) in enumerate(TRIPLE_DERIV_DIAG):
    aow_triple = (np.einsum("gu, g -> gu", ao[i3x], wv[1])
                + np.einsum("gu, g -> gu", ao[i3y], wv[2])
                + np.einsum("gu, g -> gu", ao[i3z], wv[3]))
    dao_vxc_diag[idx] += 2 * np.einsum("gu, gu -> u", aow_triple, ao_dm0[0])

de_vxc_diag = np.zeros((natm, natm, 6))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    de_vxc_diag[A, A] += np.einsum("Au -> A", dao_vxc_diag[:, slcA])
de_vxc_diag = de_vxc_diag[:, :, [[0, 1, 2], [1, 3, 4], [2, 4, 5]]]
print("de_vxc_diag fp:", lib.fp(de_vxc_diag))

de_vxc_diag fp: 49.68876638573063


### dao_vxc_off contribution

In [17]:
# --- dao_vxc (GGA: no tau) --- #
wv = weights * vxc  # [4, ngrids]
dao_vxc = np.zeros((3, 3, nao, nao))

# GGA part (RHO + SIGMA)
GGA_CALLS = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]

aowv = [None, None, None]
for t in range(3):
    aowv[t] = 0.5 * np.einsum("gu, g -> gu", ao[t + 1], wv[0])
    for r in range(3):
        aowv[t] += np.einsum("gu, g -> gu", ao[GGA_CALLS[t][r]], wv[r + 1])

for t in range(3):
    for s in range(3):
        dao_vxc[t, s] += 2 * aowv[s].T @ ao[t + 1]     # ipip[t,s]

dao_vxc += dao_vxc.transpose(1, 0, 3, 2)  # [s,t] with AO indices transposed

de_vxc_off = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    for B in range(A + 1):
        _, _, p0B, p1B = aoslices[B]
        slcB = slice(p0B, p1B)
        de_vxc_off[A, B] += np.einsum("tsuv, uv -> ts", dao_vxc[:, :, slcB, slcA], dm0[slcB, slcA])
        if A != B:
            de_vxc_off[B, A] = de_vxc_off[A, B].T
print("de_vxc fp:", lib.fp(de_vxc_off))

de_vxc fp: -29.337474734527564


### summarize of common DFT contribution

In [18]:
de_xc_recap = de_vxc_diag + de_vxc_off + de_fxc
assert np.allclose(de_xc_recap, de_ks_ref)

In [19]:
dat = dict(np.load("nh3_r_b3lyp_decomp.npz"))
dat.update({
    "de_vxc_diag": de_vxc_diag,
    "de_vxc_off": de_vxc_off,
    "de_fxc": de_fxc,
})
np.savez("nh3_r_b3lyp_decomp.npz", **dat)

In [20]:
de_xc_recap.sum(axis=(0, 1))

array([[ 0.0004 , -0.00001,  0.     ],
       [-0.00001,  0.00034,  0.00003],
       [ 0.     ,  0.00003,  0.00037]])

In [21]:
np.abs(de_xc_recap.sum(axis=(0, 1))).max()

np.float64(0.0003977856045045136)

### becke partition derivative (preparation)

In [22]:
ngrids = grids.weights.size
ni = dft.numint.NumInt()
ao = ni.eval_ao(mol, grids.coords, deriv=3)   # deriv=3: needed for d2rho (t4/t7)
dm0 = mf.make_rdm1()
ao_dm0 = ao @ dm0
rho, exc, vxc, fxc = rks_nimatmul._eval_rho_exc_vxc_fxc("B3LYP", "GGA", ao, ao_dm0)
drho = rks_nimatmul._make_drho("GGA", ao, ao_dm0, mol.aoslice_by_atom())

In [23]:
natm = mol.natm
becke_scheme = grids.radii_adjust(mol, grids.atomic_radii)
adjustment_factor = np.array([becke_scheme(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)
becke_result = becke_partition(grids.coords, mol.atom_coords(), grids.atm_idx, grids.quadrature_weights, adjustment_factor, 3, 512, 2, None)
w, dw, ddw = becke_result["w"], becke_result["dw"], becke_result["ddw"]

In [24]:
# Second-order skeleton density derivative d2rho[C, s, t, x, g] = d/dr_t of drho[C, s, x, g].
# (drho is the first skeleton derivative d rho_x / d R_{C_s}; d2rho takes one more spatial deriv.)
# Needed for the d(d rho/d r)/dB term (t4) and the double grid-shift (t7).
IDX2 = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]
IDX3 = [
    [[XXX, XXY, XXZ], [XXY, XYY, XYZ], [XXZ, XYZ, XZZ]],
    [[XXY, XYY, XYZ], [XYY, YYY, YYZ], [XYZ, YYZ, YZZ]],
    [[XXZ, XYZ, XZZ], [XYZ, YYZ, YZZ], [XZZ, YZZ, ZZZ]],
]
d2rho = np.zeros((natm, 3, 3, 4, ngrids))
for C in range(natm):
    _, _, p0, p1 = aoslices[C]
    slc = slice(p0, p1)
    ao_slc = ao[:, :, slc]
    ao_dm0_slc = ao_dm0[:, :, slc]
    # x = 0 (rho value)
    for s in range(3):
        for t in range(3):
            term = (np.einsum("gu, gu -> g", ao_slc[IDX2[t][s]], ao_dm0_slc[O])
                  + np.einsum("gu, gu -> g", ao_slc[s + 1], ao_dm0_slc[t + 1]))
            d2rho[C, s, t, 0] -= 2 * term
    # x = k+1 (sigma gradient component); needs 3rd-order AO derivatives
    for k in range(3):
        for s in range(3):
            for t in range(3):
                term = (np.einsum("gu, gu -> g", ao_slc[IDX3[t][s][k]], ao_dm0_slc[O])
                      + np.einsum("gu, gu -> g", ao_slc[IDX2[s][k]], ao_dm0_slc[t + 1])
                      + np.einsum("gu, gu -> g", ao_slc[IDX2[t][s]], ao_dm0_slc[k + 1])
                      + np.einsum("gu, gu -> g", ao_slc[s + 1], ao_dm0_slc[IDX2[t][k]]))
                d2rho[C, s, t, k + 1] -= 2 * term
d2rho_sum = d2rho.sum(axis=0)  # (s, t, x, g) = -d2 rho_x / (d r_s d r_t)

## 格点偏移二阶导数分解

回顾一阶梯度（`11-1`）中，格点偏移梯度分为两部分，且两部分大小相近、符号相反，加起来接近零（但精确描述了格点偏移增量）：

- 格点权重梯度 $T_1[A,t] = \sum_g \frac{dw_g}{dA_t} f_g$
- 泛函对格点偏移梯度 $T_2[A,t] = \sum_{g \in A} w_g\, vxc_x\, \frac{\partial \rho_x}{\partial r_{tg}}$

二阶情形完全类似。记 $dT_1 = d(T_1)/dB_s$、$dT_2 = d(T_2)/dB_s$，则格点偏移 Hessian 增量

$$\Delta H_{AB,ts} = dT_1[A,B,t,s] + dT_2[A,B,t,s]$$

两部分同样大小相近、符号相反，加起来给出小的格点偏移增量（$\sim 10^{-3}$），用以修正 `de_xc_recap` 的平动不变性（其 $\sum_{AB}$ 当前约 $4\times10^{-4}$）。

由于格点是随原子移动的，$dT_1, dT_2$ 的每一项都要做 `+= transpose` 对称化——这恰好把“骨架梯度随格点移动”的那部分贡献自动补上（除了 $A=B$ 的二阶格点偏移项 `t7`，需单独添加）。下面各项 $t_1,\dots,t_7$ 均按此约定。

### t1

In [25]:
t1 = np.einsum("Atg, xg, Bsxg -> ABts", dw, vxc, drho)
t1 += np.einsum("ABts -> BAst", t1)

In [26]:
t1[0, 1]

array([[-0.93322, -0.05716, -0.16004],
       [-0.08997,  0.0516 , -0.01137],
       [-0.18441, -0.0075 ,  0.01248]])

### t2

In [27]:
t2 = np.einsum("AtBsg, g, g -> ABts", ddw, exc, rho[0])

In [28]:
t2[0, 1]

array([[ 0.001  , -0.1181 , -0.13659],
       [ 0.00622,  0.30383, -0.00699],
       [-0.0326 , -0.02132,  0.3082 ]])

### t3

In [29]:
t3 = np.zeros((natm, natm, 3, 3))

dsum_rho = drho.sum(axis=0)
for A in range(natm):
    maskA = grids.atm_idx == A
    t3[A] -= np.einsum("g, txg, xyg, Bsyg -> Bts", weights[maskA], dsum_rho[..., maskA], fxc[..., maskA], drho[..., maskA])
t3 += np.einsum("ABts -> BAst", t3)

In [30]:
t3[0, 1]

array([[-0.02617, -0.04418, -0.05123],
       [ 0.01006,  0.08572, -0.00139],
       [-0.00921, -0.00793,  0.08664]])

### t4

In [31]:
# t4 = c3: the d(d rho/d r)/dB term of dT2.
#   T2 carries a factor (d rho_x / d r_{tg}); differentiating it w.r.t. B_s gives
#   -sum_{g in A} w_g vxc_x d(d rho_sum[t,x])/dB_s
#         = sum_{g in A} w_g vxc_x * d2rho[B, s, t, x, g]   (A != B)
#   For A == B the grid owned by A also moves with B, so use (d2rho[A] - d2rho_sum)
#   (= -sum_{C != A} d2rho[C]); the missing A=B piece is restored by t7 below.
t4 = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    maskA = grids.atm_idx == A
    wA = w[maskA]
    vxcA = vxc[:, maskA]
    for B in range(natm):
        if B != A:
            t4[A, B] += np.einsum("g, xg, gstx -> ts", wA, vxcA, d2rho[B, :, :, :, maskA])
        else:
            t4[A, A] += np.einsum("g, xg, stxg -> ts", wA, vxcA, (d2rho[A] - d2rho_sum)[:, :, :, maskA])

In [32]:
# Symmetrise: += transpose.  As for t1/t3/t5, this also adds the
# "skeleton-gradient carried by the moving grid" contribution (the transpose half).
t4 += np.einsum("ABts -> BAst", t4)

In [33]:
t4[0, 1]

array([[ 0.95943,  0.1013 ,  0.21125],
       [ 0.07989, -0.13728,  0.01276],
       [ 0.19363,  0.01542, -0.09915]])

### t5

In [34]:
t5 = np.zeros((natm, natm, 3, 3))

dsum_rho = drho.sum(axis=0)
for A in range(natm):
    maskA = grids.atm_idx == A
    t5[A] -= np.einsum("Bsg, xg, txg -> Bts", dw[..., maskA], vxc[..., maskA], dsum_rho[..., maskA])
t5 += np.einsum("ABts -> BAst", t5)

In [35]:
t5[0, 1]

array([[-0.00114,  0.11828,  0.13659],
       [-0.00597, -0.30408,  0.00691],
       [ 0.0325 ,  0.02132, -0.3081 ]])

In [36]:
(t2 + t5)[0, 1]

array([[-0.00014,  0.00017, -0.00001],
       [ 0.00024, -0.00026, -0.00009],
       [-0.0001 , -0.     ,  0.00009]])

### t6

In [37]:
# t6 = c2b: the vxc-derivative grid-shift part of dT2 (only A == B, where the
# grid owned by A also moves with B). Cancels the A=B tail of t3 (huge, opposite sign).
t6 = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    maskA = grids.atm_idx == A
    t6[A, A] += np.einsum("g, xyg, syg, txg -> ts", w[maskA], fxc[..., maskA],
                          dsum_rho[..., maskA], dsum_rho[..., maskA])

In [38]:
t6[0, 0]

array([[-102.89431,   -0.10105,    0.06761],
       [  -0.10105, -103.32008,   -0.02232],
       [   0.06761,   -0.02232, -103.45325]])

### t7

In [39]:
# t7 = extra: the missing A == B double grid-shift.
# Symmetrising t4 (c3) adds c3^T, but for A == B the true grid-motion piece g3
# differs from c3^T by sum_{g in A} w_g vxc_x d2rho_sum[t, s, x, g]; that residual is t7.
# (By symmetry of mixed partials, d2rho_sum[s, t] = d(d rho_sum[t])/d r_s.)
t7 = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    maskA = grids.atm_idx == A
    t7[A, A] += np.einsum("g, xg, stxg -> ts", w[maskA], vxc[:, maskA], d2rho_sum[:, :, :, maskA])

In [40]:
t7[0, 0]

array([[-103.27546,   -0.10877,    0.06439],
       [  -0.10877, -103.58396,   -0.01933],
       [   0.06439,   -0.01933, -103.68833]])

### assemble grid-shift hessian

In [41]:
# dT1 = t1 + t2  (from the grid-weight gradient T1)
# dT2 = t3 + t4 + t5 + t6 + t7  (from the functional grid-shift gradient T2)
# The two are individually large but opposite, summing to the small grid-shift increment
# (the hessian analogue of the gradient's T1 + T2 cancellation).
dT1 = t1 + t2
dT2 = t3 + t4 + t5 + t6 + t7
grid_shift = dT1 + dT2
print("dT1 max:", np.abs(dT1).max())
print("dT2 max:", np.abs(dT2).max())
print("grid_shift max (deviation from de_xc_recap):", np.abs(grid_shift).max())
print("grid_shift sum(0,1) max:", np.abs(grid_shift.sum(axis=(0, 1))).max())

dT1 max: 0.9398088411206892
dT2 max: 0.9401093792017292
grid_shift max (deviation from de_xc_recap): 0.0007082207716656574
grid_shift sum(0,1) max: 0.0003977856038858765


### cancellation check: t2 + t5 (non-diagonal)

In [42]:
# As in the gradient, the weight pair t2 + t5 is individually large but cancels
# to ~1e-4 on non-diagonal atom pairs (the user's hint).
print("(t2 + t5)[0, 1]:"); print((t2 + t5)[0, 1])
print()
print("max|(t2+t5)[A,B]| over A != B:",
      max(np.abs(t2[A, B] + t5[A, B]).max() for A in range(natm) for B in range(natm) if A != B))

(t2 + t5)[0, 1]:
[[-0.00014  0.00017 -0.00001]
 [ 0.00024 -0.00026 -0.00009]
 [-0.0001  -0.       0.00009]]

max|(t2+t5)[A,B]| over A != B: 0.00025515106407580923


### translational invariance: de_xc_recap + grid_shift

In [43]:
# Adding the grid-shift increment should restore translational invariance:
# sum over (A, B) of the DFT skeleton hessian drops from ~4e-4 to ~1e-13.
de_xc_grid = de_xc_recap + grid_shift
print("|de_xc_recap|.sum(0,1) max            :", np.abs(de_xc_recap.sum(axis=(0, 1))).max())
print("|de_xc_recap + grid_shift|.sum(0,1) max:", np.abs(de_xc_grid.sum(axis=(0, 1))).max())
print()
print("(np.allclose(de_xc_recap, de_xc_grid) will fail by ~1e-3, which is expected;")
print(" the grid-shift is a small correction of that magnitude.)")

|de_xc_recap|.sum(0,1) max            : 0.0003977856045045136
|de_xc_recap + grid_shift|.sum(0,1) max: 7.69648234033582e-13

(np.allclose(de_xc_recap, de_xc_grid) will fail by ~1e-3, which is expected;
 the grid-shift is a small correction of that magnitude.)


### finite-difference check (muted; call `run_fd_check()` to run)

In [44]:
# Finite-difference check (COSTLY: 24 displaced geometries, ~1 min).
# Wrapped in a function so the notebook does not run it eagerly; call when needed.
#
# FDs the full grid-response gradient G^total = G^skel + T1 + T2 (grid rebuilt at each
# displaced geometry, dm0 held fixed = skeleton, no SCF). Its B-derivative is the full
# grid-moving skeleton hessian, which must equal de_ks_ref + grid_shift.
# (Equivalently, grid_shift = FD(G^total) - de_ks_ref = [FD(G^skel)-de_ks_ref] + FD(T1+T2),
#  i.e. the grid-motion-of-G^skel piece plus d/dB[T1+T2].)
def grid_response_grad(m, dm0):
    g = dft.gen_grid.Grids(m); g.build(sort_grids=False)
    ni_ = dft.numint.NumInt()
    ao_ = ni_.eval_ao(m, g.coords, deriv=2)
    ao_dm0_ = ao_ @ dm0
    rho_, exc_, vxc_, fxc_ = rks_nimatmul._eval_rho_exc_vxc_fxc("B3LYP", "GGA", ao_, ao_dm0_)
    drho_ = rks_nimatmul._make_drho("GGA", ao_, ao_dm0_, m.aoslice_by_atom())
    bs_ = g.radii_adjust(m, g.atomic_radii)
    adj_ = np.array([bs_(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)
    br_ = becke_partition(g.coords, m.atom_coords(), g.atm_idx, g.quadrature_weights, adj_, 3, 512, 2, None)
    w_, dw_ = br_["w"], br_["dw"]
    Gskel = np.einsum("g, xg, Atxg -> At", w_, vxc_, drho_)            # skeleton gradient
    T1 = np.einsum("Atg, g, g -> At", dw_, exc_, rho_[0])              # grid-weight gradient
    dsum = drho_.sum(axis=0)
    T2 = np.zeros((natm, 3))                                          # functional grid-shift gradient
    for A in range(natm):
        mA = g.atm_idx == A
        T2[A] -= np.einsum("g, xg, txg -> t", w_[mA], vxc_[:, mA], dsum[:, :, mA])
    return Gskel + T1 + T2


def run_fd_check(h=1e-4):
    """FD the grid-response gradient and compare to de_ks_ref + grid_shift."""
    coords0 = mol.atom_coords().copy()
    fd_total = np.zeros((natm, natm, 3, 3))
    for B in range(natm):
        for s in range(3):
            cp = coords0.copy(); cp[B, s] += h
            cm = coords0.copy(); cm[B, s] -= h
            mp = mol.copy(); mp.verbose = 0; mp.set_geom_(cp, unit="Bohr"); mp.build(False, False)
            mm = mol.copy(); mm.verbose = 0; mm.set_geom_(cm, unit="Bohr"); mm.build(False, False)
            fd_total[:, B, :, s] = (grid_response_grad(mp, dm0) - grid_response_grad(mm, dm0)) / (2 * h)
    fd_total = 0.5 * (fd_total + np.einsum("ABts -> BAst", fd_total))
    print("max|(de_ks_ref + grid_shift) - FD(G^total)|:",
          np.abs(de_ks_ref + grid_shift - fd_total).max())
    print("(should be ~1e-5, finite-difference truncation error)")
    return fd_total


# Uncomment to run (costly):
# fd_total = run_fd_check()


### per-atom split: t8 (with dao_vxc_diag) and t9 (with dao_vxc)

`t4t` (no `d2rho`) is split into two contributions the same way production splits
`dao_vxc_diag` / `dao_vxc`:

- **`t8`** = `c3_diag + g3_diag` -- generated alongside `dao_vxc_diag` (mirrors
  cell 19). Its d2rho terms (T1, U3, U1) share cell-19's `aow_diag` / `ao_dm0[0]`
  intermediates (`ip2`, `ip3`).
- **`t9`** = `c3_off + g3_off` -- generated alongside `dao_vxc` (mirrors cell 21).
  Its d2rho terms (T2, U2, U4) are the two AO-index traces of the symmetrised
  `dao_vxc`, so `Mdm0_off = 0.5 * einsum("tsab, ba -> tsa", dao_vxc_A, dm0)` --
  the dm0-contraction ("diagonal") of `dao_vxc` itself, no `cdm0v` buffer.

`t4t = t8 + t9`. Both split the grid loop by `atm_idx` (a *perfect partition* --
per-atom pieces sum back to the full-grid `dao_vxc_diag` / `dao_vxc`). `dm0` is
contracted **early** (via `ao_dm0`) into a diagonal `Mdm0[A,t,s,mu]` (`nao`, not
`nao^2`); each d2rho term lands in exactly one vecdot reusing a usual-KS
intermediate, so no `d2rho` is needed.


In [45]:
# --- dao_vxc_diag (GGA: no tau) --- #  [per-atom split; mirrors cell 19]
#
# Production splits the grid-shift Hessian (t4t = c3 + g3) the same way it splits
# dao_vxc_diag / dao_vxc: this section mirrors cell 19 and additionally yields the
# DIAGONAL half t8 = c3_diag + g3_diag -- the d2rho terms (T1, U3, U1) that share
# cell-19's aow_diag / ao_dm0[0] intermediates. The per-A pieces of dao_vxc_diag sum
# to the full-grid value (perfect partition; verified vs cell 19). dm0 is contracted
# early (via ao_dm0) so Mdm0_diag is diagonal (nao, not nao^2).
wv = weights * vxc
PAIR_TS = np.array([[0, 1, 2], [1, 3, 4], [2, 4, 5]])  # (t,s) -> symmetric-pair idx for ip6


def _c3g3(Mdm0):
    # c3[A,B] = -2 sum_{mu in B} Mdm0[A][t,s];  g3[A,B] = -2 sum_{mu in A} Mdm0[B][s,t]
    out = np.zeros((natm, natm, 3, 3))
    for A in range(natm):
        _, _, p0A, p1A = aoslices[A]; slcA = slice(p0A, p1A)
        for B in range(natm):
            _, _, p0B, p1B = aoslices[B]; slcB = slice(p0B, p1B)
            out[A, B] += -2 * np.einsum("tsu -> ts", Mdm0[A][:, :, slcB])               # c3 (sum mu in B)
            if A == B:
                out[A, A] += 2 * np.einsum("tsu -> ts", Mdm0[A])                        # c3 A=B correction
            out[A, B] += -2 * np.einsum("tsu -> ts", Mdm0[B].transpose(1, 0, 2)[:, :, slcA])  # g3 (Mdm0[B][s,t], sum mu in A)
    return out


dao_vxc_diag2 = np.zeros((6, nao))                  # per-A (= cell 19 dao_vxc_diag)
Mdm0_diag = np.zeros((natm, 3, 3, nao))             # -> t8
for A in range(natm):
    mA = grids.atm_idx == A
    aoA = ao[:, mA, :]; ao_dm0A = ao_dm0[:, mA, :]; wvA = wv[:, mA]
    ip6 = np.zeros((6, nao))                         # cell-19 contributions (before the RKS *2)
    # Contribution 1: ao[its]^T @ (wv[0]*ao_dm0[0] + wv[1]*ao_dm0[1] + wv[2]*ao_dm0[2] + wv[3]*ao_dm0[3])
    aow_diag = (np.einsum("gu, g -> gu", ao_dm0A[0], wvA[0])
              + np.einsum("gu, g -> gu", ao_dm0A[1], wvA[1])
              + np.einsum("gu, g -> gu", ao_dm0A[2], wvA[2])
              + np.einsum("gu, g -> gu", ao_dm0A[3], wvA[3]))
    for idx, its in enumerate([XX, XY, XZ, YY, YZ, ZZ]):
        ip6[idx] += np.einsum("gu, gu -> u", aoA[its], aow_diag)
    # Contribution 2 (GGA triple-derivative part)
    for idx, (i3x, i3y, i3z) in enumerate(TRIPLE_DERIV_DIAG):
        aow_triple = (np.einsum("gu, g -> gu", aoA[i3x], wvA[1])
                    + np.einsum("gu, g -> gu", aoA[i3y], wvA[2])
                    + np.einsum("gu, g -> gu", aoA[i3z], wvA[3]))
        ip6[idx] += np.einsum("gu, gu -> u", aow_triple, ao_dm0A[0])
    dao_vxc_diag2 += 2 * ip6                        # = cell 19 dao_vxc_diag (per-A sums to full)
    Mdm0_diag[A] = ip6[PAIR_TS]                     # expand 6 sym pairs -> (3,3) for t8

# verify the per-A split recovers cell 19's dao_vxc_diag
print("max|dao_vxc_diag(per-A) - dao_vxc_diag|:", np.abs(dao_vxc_diag2 - dao_vxc_diag).max())

# t8 = c3_diag + g3_diag from Mdm0_diag (d2rho terms T1, U3, U1; no d2rho used)
t8 = _c3g3(Mdm0_diag)


max|dao_vxc_diag(per-A) - dao_vxc_diag|: 2.2737367544323206e-13


In [46]:
# --- dao_vxc (GGA: no tau) --- #  [per-atom split; mirrors cell 21]
#
# Mirrors cell 21 and additionally yields the OFF-DIAGONAL half t9 = c3_off + g3_off.
# The off-diagonal part of Mdm0 is just the dm0-contraction ("diagonal") of dao_vxc
# itself:
#   Mdm0_off[A] = 0.5 * einsum("tsab, ba -> tsa", dao_vxc_A, dm0)
# because part4 (<aowv[s], ao_dm0[t+1]>) and term2 (<ao[s+1], cdm0v[t]>) are the two
# AO-index traces of the symmetrised dao_vxc matrix (raw + raw.T). So no separate
# cdm0v buffer is built: the second term is obtained from the same dao_vxc that
# cell 21 already produces.
wv = weights * vxc
dao_vxc2 = np.zeros((3, 3, nao, nao))               # per-A (= cell 21 dao_vxc)
Mdm0_off = np.zeros((natm, 3, 3, nao))              # -> t9
for A in range(natm):
    mA = grids.atm_idx == A
    aoA = ao[:, mA, :]; wvA = wv[:, mA]
    # aowv (cell 21): weighted ao
    aowv = [None, None, None]
    for t in range(3):
        aowv[t] = 0.5 * np.einsum("gu, g -> gu", aoA[t + 1], wvA[0])
        for r in range(3):
            aowv[t] += np.einsum("gu, g -> gu", aoA[GGA_CALLS[t][r]], wvA[r + 1])
    # dao_vxc (cell 21) -- per-A piece, then symmetrise [s,t] with AO indices transposed
    dao_vxc_A = np.zeros((3, 3, nao, nao))
    for t in range(3):
        for s in range(3):
            dao_vxc_A[t, s] += 2 * aowv[s].T @ aoA[t + 1]     # ipip[t,s]
    dao_vxc_A += dao_vxc_A.transpose(1, 0, 3, 2)
    dao_vxc2 += dao_vxc_A                                    # accumulate -> full-grid dao_vxc
    # Mdm0_off = 0.5 * (dm0-contraction of dao_vxc_A) -- the "diagonal" of dao_vxc
    Mdm0_off[A] = 0.5 * np.einsum("tsab, ba -> tsa", dao_vxc_A, dm0)

# verify the per-A split recovers cell 21's dao_vxc
print("max|dao_vxc(per-A) - dao_vxc|:", np.abs(dao_vxc2 - dao_vxc).max())

# t9 = c3_off + g3_off from Mdm0_off (T2, U2, U4);  t4t = t8 + t9
t9 = _c3g3(Mdm0_off)
t4t = t8 + t9
print("max|t4t - (t4 + t7)|:", np.abs(t4t - (t4 + t7)).max())


max|dao_vxc(per-A) - dao_vxc|: 5.684341886080802e-14
max|t4t - (t4 + t7)|: 5.400124791776761e-13


#### grid-shift hessian with d2rho fully substituted

In [47]:
# Reassemble the grid-shift hessian with t4+t7 replaced by the basis-form t4t.
# No d2rho (nor d2rho_sum) is used; only drho (in t1/t3/t5/t6) remains, as in production.
grid_shift_basis = t1 + t2 + t3 + t4t + t5 + t6
print("max|grid_shift_basis - grid_shift|:", np.abs(grid_shift_basis - grid_shift).max())
print("|de_ks_ref + grid_shift_basis|.sum(0,1) max:",
      np.abs((de_ks_ref + grid_shift_basis).sum(axis=(0, 1))).max())

max|grid_shift_basis - grid_shift|: 5.573874695130598e-13
|de_ks_ref + grid_shift_basis|.sum(0,1) max: 3.2940317140628395e-13
